In [1]:
# ============================================================
# MARKETPLACE REVIEW INTELLIGENCE
# Notebook 03 — Engenharia de Features via product_url
# ============================================================
# Objetivo: Extrair informações estruturadas da product_url:
# - produto_id  → identificador único do produto
# - slug        → descrição bruta extraída da URL
# - nome_produto → nome legível gerado pelo slug
# - categoria   → subcategoria inferida por palavras-chave
# ============================================================

import pandas as pd
import re
from pathlib import Path

# Caminhos do projeto
ROOT = Path().resolve().parent
PROCESSED_PATH = ROOT / "data" / "processed"

# Carrega o dataset limpo gerado no Notebook 02
df = pd.read_csv(PROCESSED_PATH / "reviews_limpos.csv", encoding="utf-8-sig")
df["date"] = pd.to_datetime(df["date"])

print(f"✅ Dataset carregado: {df.shape[0]:,} registros")
print(f"📋 Colunas: {list(df.columns)}")
print(f"\nAmostra de URLs:")
display(df["product_url"].head(5))

✅ Dataset carregado: 202,785 registros
📋 Colunas: ['date', 'rating', 'content', 'product_url', 'qtd_palavras', 'review_curto', 'content_original']

Amostra de URLs:


0    https://produto.mercadolivre.com.br/MLB-314957...
1    https://produto.mercadolivre.com.br/MLB-314957...
2    https://produto.mercadolivre.com.br/MLB-314957...
3    https://produto.mercadolivre.com.br/MLB-314957...
4    https://produto.mercadolivre.com.br/MLB-314957...
Name: product_url, dtype: object

In [2]:
# ============================================================
# Extrai o ID único do produto direto da URL
# Padrão 1: /MLB-XXXXXXXXX-slug-_JM  (produto.mercadolivre)
# Padrão 2: /slug/p/MLB19XXXXXXX     (www.mercadolivre)
# ============================================================

def extrair_produto_id(url):
    """
    Tenta extrair o ID do produto MLB da URL.
    Suporta dois formatos de URL do Mercado Livre.
    Retorna None se não encontrar.
    """
    if pd.isna(url):
        return None

    # Padrão 1: MLB- no início do path (produto.mercadolivre.com.br)
    match = re.search(r"/(MLB-\d+)-", str(url))
    if match:
        return match.group(1)

    # Padrão 2: /p/MLB no final da URL (www.mercadolivre.com.br)
    match = re.search(r"/p/(MLB\d+)", str(url))
    if match:
        return match.group(1)

    return None

df["produto_id"] = df["product_url"].apply(extrair_produto_id)

# Diagnóstico
total_extraidos = df["produto_id"].notna().sum()
total_nao_extraidos = df["produto_id"].isna().sum()

print(f"✅ IDs extraídos com sucesso: {total_extraidos:,}")
print(f"⚠️  IDs não encontrados:      {total_nao_extraidos:,}")
print(f"\nExemplos de IDs extraídos:")
display(df[["product_url", "produto_id"]].dropna().head(8))

✅ IDs extraídos com sucesso: 202,783
⚠️  IDs não encontrados:      2

Exemplos de IDs extraídos:


,product_url,produto_id
0,https://produto.mercadolivre.com.br/MLB-314957...,MLB-3149572356
1,https://produto.mercadolivre.com.br/MLB-314957...,MLB-3149572356
2,https://produto.mercadolivre.com.br/MLB-314957...,MLB-3149572356
3,https://produto.mercadolivre.com.br/MLB-314957...,MLB-3149572356
4,https://produto.mercadolivre.com.br/MLB-314957...,MLB-3149572356
5,https://produto.mercadolivre.com.br/MLB-314885...,MLB-3148854797
6,https://produto.mercadolivre.com.br/MLB-314885...,MLB-3148854797
7,https://produto.mercadolivre.com.br/MLB-314885...,MLB-3148854797


In [8]:
# ============================================================
# Extrai o slug descritivo da URL
# O slug é a parte com o nome do produto separado por hífens
# ============================================================

def extrair_slug(url):
    """
    Extrai o slug do produto da URL do Mercado Livre.
    Suporta os dois formatos de URL.
    Remove o ID MLB e sufixos como _JM, /p/, parâmetros etc.
    """
    if pd.isna(url):
        return None

    url_str = str(url)

    # Remove parâmetros de query string
    url_str = url_str.split("?")[0]

    # Padrão 1: produto.mercadolivre.com.br/MLB-ID-slug-_JM
    match = re.search(r"/MLB-\d+-(.+?)(?:-_JM)?/?$", url_str)
    if match:
        return match.group(1).strip("-")

    # Padrão 2: www.mercadolivre.com.br/slug/p/MLBID
    match = re.search(r"\.com\.br/(.+?)/p/MLB", url_str)
    if match:
        return match.group(1).strip("-")

    return None

df["slug"] = df["product_url"].apply(extrair_slug)

print(f"✅ Slugs extraídos: {df['slug'].notna().sum():,}")
print(f"\nExemplos de slugs:")
display(df[["product_url", "slug"]].dropna().head(8))

✅ Slugs extraídos: 202,783

Exemplos de slugs:


,product_url,slug
0,https://produto.mercadolivre.com.br/MLB-314957...,shampoo-prevent-cetoconazol-anticaspa-e-coceir...
1,https://produto.mercadolivre.com.br/MLB-314957...,shampoo-prevent-cetoconazol-anticaspa-e-coceir...
2,https://produto.mercadolivre.com.br/MLB-314957...,shampoo-prevent-cetoconazol-anticaspa-e-coceir...
3,https://produto.mercadolivre.com.br/MLB-314957...,shampoo-prevent-cetoconazol-anticaspa-e-coceir...
4,https://produto.mercadolivre.com.br/MLB-314957...,shampoo-prevent-cetoconazol-anticaspa-e-coceir...
5,https://produto.mercadolivre.com.br/MLB-314885...,selante-let-me-be-1-litro-cambuca-e-pincelluva
6,https://produto.mercadolivre.com.br/MLB-314885...,selante-let-me-be-1-litro-cambuca-e-pincelluva
7,https://produto.mercadolivre.com.br/MLB-314885...,selante-let-me-be-1-litro-cambuca-e-pincelluva


In [10]:
# ============================================================
# Converte o slug em nome legível
# Ex: "shampoo-prevent-cetoconazol-anticaspa" 
#   → "Shampoo Prevent Cetoconazol Anticaspa"
# ============================================================

def slug_para_nome(slug):
    """
    Converte slug com hífens em nome legível com Title Case.
    Remove tokens numéricos isolados e sufixos irrelevantes.
    """
    if pd.isna(slug):
        return None

    # Substitui hífens por espaços
    nome = str(slug).replace("-", " ")

    # Remove sufixos irrelevantes do Mercado Livre
    nome = re.sub(r"\b(jm|_jm|com|br)\b", "", nome, flags=re.IGNORECASE)

    # Remove espaços extras
    nome = re.sub(r"\s+", " ", nome).strip()

    # Title Case
    return nome.title()

df["nome_produto"] = df["slug"].apply(slug_para_nome)

print(f"✅ Nomes gerados: {df['nome_produto'].notna().sum():,}")
print(f"\nComparação slug → nome:")
display(df[["slug", "nome_produto"]].dropna().head(8))

✅ Nomes gerados: 202,783

Comparação slug → nome:


,slug,nome_produto
0,shampoo-prevent-cetoconazol-anticaspa-e-coceir...,Shampoo Prevent Cetoconazol Anticaspa E Coceir...
1,shampoo-prevent-cetoconazol-anticaspa-e-coceir...,Shampoo Prevent Cetoconazol Anticaspa E Coceir...
2,shampoo-prevent-cetoconazol-anticaspa-e-coceir...,Shampoo Prevent Cetoconazol Anticaspa E Coceir...
3,shampoo-prevent-cetoconazol-anticaspa-e-coceir...,Shampoo Prevent Cetoconazol Anticaspa E Coceir...
4,shampoo-prevent-cetoconazol-anticaspa-e-coceir...,Shampoo Prevent Cetoconazol Anticaspa E Coceir...
5,selante-let-me-be-1-litro-cambuca-e-pincelluva,Selante Let Me Be 1 Litro Cambuca E Pincelluva
6,selante-let-me-be-1-litro-cambuca-e-pincelluva,Selante Let Me Be 1 Litro Cambuca E Pincelluva
7,selante-let-me-be-1-litro-cambuca-e-pincelluva,Selante Let Me Be 1 Litro Cambuca E Pincelluva


In [15]:
# ============================================================
# MARKETPLACE REVIEW INTELLIGENCE
# Notebook 03 — Bloco 5 REVISADO
# ============================================================
# Dicionário atualizado após investigação dos registros
# em 'Outros Cosméticos' — versão 2.0
#
# Ajustes realizados:
# 1. Nova subcategoria: Creme Capilar
# 2. Selante adicionado à Selagem / Nanoplastia
# 3. Instant Hair e Fibra Capilar → Finalizador / Creme
# 4. Gummy adicionado ao Suplemento Capilar
# ============================================================

CATEGORIAS = [
    # --------------------------------------------------------
    # MAIS ESPECÍFICAS PRIMEIRO — evita conflito entre categorias
    # --------------------------------------------------------

    ("Escova Progressiva",    ["progressiva", "escova", "alisante",
                               "liss", "liso", "bioliso", "defrizante",
                               "alisamento", "formolfree"]),

    ("BTX / Botox Capilar",   ["btx", "btox", "bbtx", "botox",
                               "ztox", "qatox", "bbxx"]),

    ("Tintura / Matizador",   ["matizador", "tintura", "coloracao",
                               "cor-de-cabelo", "cinza", "blond",
                               "loiro", "grisalho", "escurecedor",
                               "matizante"]),

    ("Acidificante",          ["acidificante", "acidgold",
                               "ph-balancer", "antiporosidade",
                               "neutraliza", "neutralizante",
                               "acidificando"]),

    ("Antiqueda / Tônico",    ["antiqueda", "tonico", "tnico",
                               "crescimento", "calvicie", "barba",
                               "sobrancelha", "fortificante",
                               "anticaspa", "antiqueda"]),

    ("Selagem / Nanoplastia", ["selagem", "nano", "nanoplastia",
                               "gloss", "selante", "semi-definitiva",
                               "semidefintiva"]),

    ("Sérum / Leave-in",      ["serum", "leave-in", "leave in",
                               "reparador-de-pontas", "reparador pontas",
                               "finalizador", "reparador"]),

    ("Ampola / Tratamento",   ["ampola", "reconstrutor",
                               "reconstrutora", "reestruturador",
                               "tratamento", "restaurador"]),

    ("Óleo Capilar",          ["oleo", "oil", "argan", "ricino",
                               "macadamia", "abacate", "vegetal",
                               "queratina-oleo", "oleo-capilar"]),

    ("Kit Capilar",           ["kit", "combo", "conjunto",
                               "duo", "trio", "completo"]),

    ("Creme Capilar",         ["creme-capilar", "creme capilar",
                               "umidificante", "creme-hidratante",
                               "creme-nutritivo", "creme-de-tratamento",
                               "hidratante-capilar"]),

    ("Máscara Capilar",       ["mascara", "mask", "cronograma",
                               "hidratacao", "hidratante"]),

    ("Mousse / Spray",        ["mousse", "spray", "espuma",
                               "modelador", "fixador", "aerossol"]),

    ("Finalizador / Creme",   ["creme-de-pentear", "pentear",
                               "gelatina", "geleia",
                               "ativador-de-cachos", "cachos",
                               "instant-hair", "fibra-capilar",
                               "fibra capilar", "brilho"]),

    ("Condicionador",         ["condicionador", "conditioner",
                               "co-wash", "balsamo"]),

    ("Shampoo",               ["shampoo", "xampu"]),

    ("Protetor Solar",        ["protetor-solar", "protetor solar",
                               "fps", "sunscreen", "filtro-solar",
                               "bronze", "acelerador-de-bronze",
                               "acelerador bronze"]),

    ("Perfume / Fragrância",  ["perfume", "edp", "edt",
                               "eau-de", "colonia", "fragrancia"]),

    ("Suplemento Capilar",    ["vitamina", "suplemento", "capsula",
                               "caps", "biotina", "colageno", "gummy",
                               "gomas", "hair-vitamin"]),

    ("Outros Cosméticos",     []),  # fallback — sempre por último
]

def inferir_categoria(texto):
    """
    Percorre o dicionário de categorias em ordem de prioridade.
    Retorna a primeira categoria cujas palavras-chave
    forem encontradas no slug do produto.
    Ordem importa: mais específicas vêm primeiro.
    """
    if pd.isna(texto):
        return "Outros Cosméticos"

    texto_lower = str(texto).lower()

    for categoria, palavras_chave in CATEGORIAS:
        if not palavras_chave:  # fallback
            return categoria
        for palavra in palavras_chave:
            if palavra in texto_lower:
                return categoria

    return "Outros Cosméticos"

# Aplica no slug
df["categoria"] = df["slug"].apply(inferir_categoria)

# --------------------------------------------------------
# Diagnóstico completo
# --------------------------------------------------------
print("=" * 55)
print("DISTRIBUIÇÃO POR CATEGORIA — versão 2.0")
print("=" * 55)

dist_categoria = df["categoria"].value_counts()
pct_categoria  = (dist_categoria / len(df) * 100).round(2)

diagnostico_cat = pd.DataFrame({
    "Quantidade": dist_categoria,
    "% do Total": pct_categoria
})
print(diagnostico_cat)

outros     = df[df["categoria"] == "Outros Cosméticos"]
pct_outros = round(len(outros) / len(df) * 100, 2)

print(f"\n⚠️  Registros em 'Outros Cosméticos': {len(outros):,} ({pct_outros}%)")

# Meta: manter abaixo de 10%
if pct_outros <= 10:
    print("✅ Dentro do limite aceitável (≤ 10%)")
else:
    print("❌ Acima do limite — considerar novo refinamento")

print(f"\nExemplos ainda não categorizados:")
display(outros[["slug", "nome_produto"]].head(10))

DISTRIBUIÇÃO POR CATEGORIA — versão 2.0
                       Quantidade  % do Total
categoria                                    
Escova Progressiva          31281       15.43
Kit Capilar                 22084       10.89
Máscara Capilar             17606        8.68
Óleo Capilar                14625        7.21
Antiqueda / Tônico          12920        6.37
Shampoo                     12653        6.24
Sérum / Leave-in            12260        6.05
Protetor Solar              11561        5.70
Outros Cosméticos           11385        5.61
Ampola / Tratamento          9447        4.66
Tintura / Matizador          8895        4.39
Finalizador / Creme          8486        4.18
Perfume / Fragrância         5568        2.75
Condicionador                5257        2.59
Selagem / Nanoplastia        4770        2.35
Mousse / Spray               4360        2.15
BTX / Botox Capilar          4107        2.03
Acidificante                 2665        1.31
Suplemento Capilar           2003       

,slug,nome_produto
138,queravit-cabelos-danificados-4-itens-bioextratus,Queravit Cabelos Danificados 4 Itens Bioextratus
139,queravit-cabelos-danificados-4-itens-bioextratus,Queravit Cabelos Danificados 4 Itens Bioextratus
140,queravit-cabelos-danificados-4-itens-bioextratus,Queravit Cabelos Danificados 4 Itens Bioextratus
168,ccrp-carvo-ativado-robson-peluquero-home-care-...,Ccrp Carvo Ativado Robson Peluquero Home Care ...
169,ccrp-carvo-ativado-robson-peluquero-home-care-...,Ccrp Carvo Ativado Robson Peluquero Home Care ...
170,ccrp-carvo-ativado-robson-peluquero-home-care-...,Ccrp Carvo Ativado Robson Peluquero Home Care ...
171,ccrp-carvo-ativado-robson-peluquero-home-care-...,Ccrp Carvo Ativado Robson Peluquero Home Care ...
172,ccrp-carvo-ativado-robson-peluquero-home-care-...,Ccrp Carvo Ativado Robson Peluquero Home Care ...
215,gel-cola-pierry-lohan-caixa-24-unidades-250g-g...,Gel Cola Pierry Lohan Caixa 24 Unidades 250G G...
216,gel-cola-pierry-lohan-caixa-24-unidades-250g-g...,Gel Cola Pierry Lohan Caixa 24 Unidades 250G G...


In [16]:
# ============================================================
# Validação completa das features extraídas
# ============================================================

print("=" * 55)
print("RELATÓRIO DE ENGENHARIA DE FEATURES")
print("=" * 55)

print(f"\n🔑 PRODUTO_ID")
print(f"  Extraídos:      {df['produto_id'].notna().sum():,}")
print(f"  Não extraídos:  {df['produto_id'].isna().sum():,}")
print(f"  IDs únicos:     {df['produto_id'].nunique():,}")

print(f"\n🔤 SLUG")
print(f"  Extraídos:      {df['slug'].notna().sum():,}")
print(f"  Não extraídos:  {df['slug'].isna().sum():,}")

print(f"\n🏷️  NOME_PRODUTO")
print(f"  Gerados:        {df['nome_produto'].notna().sum():,}")
print(f"  Nomes únicos:   {df['nome_produto'].nunique():,}")

print(f"\n📦 CATEGORIA")
print(f"  Categorias únicas: {df['categoria'].nunique()}")
print(f"  Distribuição:")
print(df["categoria"].value_counts().to_string())

print(f"\n✅ Shape final do dataset: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"\n📋 Todas as colunas:")
for col in df.columns:
    print(f"   - {col}")

RELATÓRIO DE ENGENHARIA DE FEATURES

🔑 PRODUTO_ID
  Extraídos:      202,783
  Não extraídos:  2
  IDs únicos:     17,972

🔤 SLUG
  Extraídos:      202,783
  Não extraídos:  2

🏷️  NOME_PRODUTO
  Gerados:        202,783
  Nomes únicos:   16,881

📦 CATEGORIA
  Categorias únicas: 20
  Distribuição:
categoria
Escova Progressiva       31281
Kit Capilar              22084
Máscara Capilar          17606
Óleo Capilar             14625
Antiqueda / Tônico       12920
Shampoo                  12653
Sérum / Leave-in         12260
Protetor Solar           11561
Outros Cosméticos        11385
Ampola / Tratamento       9447
Tintura / Matizador       8895
Finalizador / Creme       8486
Perfume / Fragrância      5568
Condicionador             5257
Selagem / Nanoplastia     4770
Mousse / Spray            4360
BTX / Botox Capilar       4107
Acidificante              2665
Suplemento Capilar        2003
Creme Capilar              852

✅ Shape final do dataset: 202,785 linhas × 11 colunas

📋 Todas as coluna

In [17]:
# ============================================================
# Salva o dataset enriquecido para o Notebook 04 (NLP)
# ============================================================

caminho_saida = PROCESSED_PATH / "reviews_enriquecidos.csv"

df.to_csv(caminho_saida, index=False, encoding="utf-8-sig")

print(f"✅ Dataset enriquecido exportado com sucesso!")
print(f"📁 Caminho: {caminho_saida}")
print(f"📊 Shape final: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

✅ Dataset enriquecido exportado com sucesso!
📁 Caminho: D:\GITHUB\portfolio-analista-dados\projetos\marketplace-review-intelligence\data\processed\reviews_enriquecidos.csv
📊 Shape final: 202,785 linhas × 11 colunas


In [ ]:
# Análise das palavras mais frequentes em 'Outros Cosméticos'
outros = df[df["categoria"] == "Outros Cosméticos"]
palavras = outros["slug"].dropna().str.replace("-", " ").str.split()
from collections import Counter
contagem = Counter([p for lista in palavras for p in lista])
print(pd.DataFrame(contagem.most_common(30), columns=["Palavra", "Frequência"]))

        Palavra  Frequência
0            de        3939
1         creme        1995
2     borabella        1013
3             e         993
4            ml         992
5           1kg         945
6       capilar         937
7          salo         768
8         250ml         711
9           bio         669
10            1         656
11         para         647
12        300ml         627
13       beleza         607
14         hair         604
15        truss         604
16      prohall         562
17            a         560
18       volume         553
19        500ml         545
20    hidrataco         536
21       cabelo         520
22          com         512
23           1l         497
24  reconstruco         486
25          one         474
26      termico         464
27            4         458
28     extratus         453
29         500g         451
